# Практична робота №11 — Метаевристичні методи глобальної оптимізації

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою тестову функцію, розмірність, розмір популяції та коефіцієнт охолодження α автоматично.

In [ ]:
import time
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # реєструє 3D-проєкцію
from scipy.optimize import minimize

In [ ]:
from variants import VARIANTS, FUNCTIONS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Модель варіанта

З таблиці варіантів дістаємо ключ тестової функції, розмірність n, розмір популяції (використовується і для генетичного алгоритму, і для рою частинок) та коефіцієнт геометричного охолодження α для імітації відпалу. `FUNCTIONS[key]` містить саму функцію, межі області пошуку та довідкові значення глобального мінімуму.

In [ ]:
FUNC_KEY, DIM, POP, ALPHA = VARIANTS[VARIANT]
INFO = FUNCTIONS[FUNC_KEY]
f = INFO["f"]
LO, HI = INFO["lo"], INFO["hi"]
F_STAR = INFO["f_star"](DIM)
X_STAR = INFO["x_star"](DIM)
SUCCESS_TOL = 0.5  # запуск вважаємо "успішним", якщо |f_best - f*| < SUCCESS_TOL

print(f"Варіант {VARIANT}: функція {INFO['label']}, n={DIM}, розмір популяції={POP}, alpha={ALPHA}")
print(f"Межі області: xi in [{LO}, {HI}]")
print(f"Глобальний мінімум: f* = {F_STAR}, x* = {X_STAR}")

## 2. Графік тестової функції

Для n=2 будуємо і поверхню, і лінії рівня. Рівні лінії рівня беремо за квантилями значень функції (а не рівномірно) — на мультимодальних функціях з великим розкидом значень рівномірні рівні ховають тонку структуру локальних мінімумів в одній-двох суцільних плямах.

In [ ]:
def contour_levels(Z, n=28):
    qs = np.linspace(0, 1, n)
    return np.unique(np.quantile(Z, qs))


if DIM == 2:
    xs = np.linspace(LO, HI, 300)
    ys = np.linspace(LO, HI, 300)
    Xg, Yg = np.meshgrid(xs, ys)
    Zg = INFO["f_grid"](Xg, Yg)

    fig_land = plt.figure(figsize=(12, 5))
    ax3d = fig_land.add_subplot(1, 2, 1, projection="3d")
    ax3d.plot_surface(Xg, Yg, Zg, cmap="viridis", linewidth=0, antialiased=True, rcount=80, ccount=80)
    ax3d.set_xlabel("x1"); ax3d.set_ylabel("x2"); ax3d.set_zlabel("f")
    ax3d.set_title(f"Поверхня f(x1, x2) - {INFO['label']}")

    axc = fig_land.add_subplot(1, 2, 2)
    axc.contour(Xg, Yg, Zg, levels=contour_levels(Zg), cmap="viridis")
    if X_STAR is not None:
        axc.plot(*X_STAR[:2], "r*", ms=16, label="x* (глобальний мін.)")
        axc.legend()
    axc.set_xlabel("x1"); axc.set_ylabel("x2")
    axc.set_title("Лінії рівня (рівні за квантилями)")
    fig_land.suptitle(f"Варіант {VARIANT}: тестова функція {INFO['label']} (n=2)")
    fig_land.tight_layout()
    fig_land.savefig("landscape_base.png", dpi=150)
    fig_land

## 3. Локальний метод (BFGS) з 20 випадкових точок

Запускаємо BFGS із 20 випадкових початкових точок в області пошуку і дивимось, у скільки різних локальних мінімумів він потрапляє. Це наочно показує, чому локальний метод непридатний для глобальної оптимізації мультимодальної функції.

In [ ]:
rng_local = np.random.default_rng(123)
local_vals = []
for _ in range(20):
    x0 = rng_local.uniform(LO, HI, DIM)
    res = minimize(f, x0, method="BFGS")
    local_vals.append(float(res.fun))
local_vals = np.array(local_vals)
local_success = int(np.sum(np.abs(local_vals - F_STAR) < SUCCESS_TOL))

fig_hist, axh = plt.subplots(figsize=(7, 4.5))
axh.hist(local_vals, bins=12, color="#4a7fb5", edgecolor="black", alpha=0.85)
axh.axvline(F_STAR, color="red", ls="--", lw=2, label=f"f* = {F_STAR:.4f}")
axh.set_xlabel("f(x*), знайдене BFGS"); axh.set_ylabel("кількість запусків")
axh.set_title(f"Варіант {VARIANT}: 20 запусків BFGS зі випадкових точок")
axh.legend()
fig_hist.tight_layout()
print(f"Успішних запусків (околиця глобального мінімуму): {local_success}/20")
fig_hist

## 4. Реалізація трьох метаевристик

Кожна функція повертає `(x*, f*, кількість_обчислень_f, історія_обчислень, історія_найкращого_значення)` — історія потрібна для графіка збіжності. Усі три реалізації будуються "з нуля" на numpy, без готових бібліотечних метаевристик.

- **Імітація відпалу**: сусідній розв'язок генерується гаусовим кроком, що масштабується з температурою; приймається за критерієм Метрополіса; температура охолоджується геометрично (Tk+1 = α·Tk).
- **Генетичний алгоритм**: турнірна селекція, арифметичний кросовер для дійсних векторів, гаусова мутація, елітизм (найкраща особина завжди переноситься в нове покоління).
- **Рій частинок (PSO)**: кожна частинка оновлює швидкість з урахуванням інерції, власного найкращого досвіду (когнітивна складова) і найкращого досвіду всього рою (соціальна складова).

In [ ]:
def simulated_annealing(f, lo, hi, n, alpha, rng, T0=100.0, Tmin=1e-4, steps=200):
    x = rng.uniform(lo, hi, n); fx = f(x)
    best, fbest = x.copy(), fx
    T, nfev = T0, 1
    hist_n, hist_b = [nfev], [fbest]
    while T > Tmin:
        for _ in range(steps):
            y = np.clip(x + rng.normal(0, T ** 0.5 / 3, n), lo, hi)
            fy = f(y); nfev += 1
            dE = fy - fx
            if dE < 0 or rng.random() < np.exp(-dE / T):  # критерій Метрополіса
                x, fx = y, fy
            if fx < fbest:
                best, fbest = x.copy(), fx
        hist_n.append(nfev); hist_b.append(fbest)
        T *= alpha  # геометричне охолодження
    return best, fbest, nfev, np.array(hist_n), np.array(hist_b)


def genetic(f, lo, hi, n, pop, rng, gens=300, pc=0.9, pm=0.1, tour=3):
    P = rng.uniform(lo, hi, (pop, n)); F = np.array([f(p) for p in P]); nfev = pop
    hist_n, hist_b = [nfev], [F.min()]
    for _ in range(gens):
        idx = rng.integers(0, pop, (pop, tour))              # турнірна селекція
        parents = P[idx[np.arange(pop), np.argmin(F[idx], axis=1)]]
        child = parents.copy()
        for i in range(0, pop - 1, 2):                        # арифметичний кросовер
            if rng.random() < pc:
                a = rng.random()
                child[i] = a * parents[i] + (1 - a) * parents[i + 1]
                child[i + 1] = a * parents[i + 1] + (1 - a) * parents[i]
        mask = rng.random((pop, n)) < pm                       # гаусова мутація
        child = np.clip(child + mask * rng.normal(0, 0.5, (pop, n)), lo, hi)
        Fc = np.array([f(c) for c in child]); nfev += pop
        worst = int(np.argmax(Fc)); best_i = int(np.argmin(F))
        child[worst], Fc[worst] = P[best_i], F[best_i]          # елітизм
        P, F = child, Fc
        hist_n.append(nfev); hist_b.append(F.min())
    k = int(np.argmin(F))
    return P[k], F[k], nfev, np.array(hist_n), np.array(hist_b)


def pso(f, lo, hi, n, swarm, rng, iters=400, w=0.72, c1=1.5, c2=1.5):
    X = rng.uniform(lo, hi, (swarm, n)); V = rng.uniform(-1, 1, (swarm, n))
    F = np.array([f(x) for x in X]); nfev = swarm
    Pb, Fpb = X.copy(), F.copy()
    g = int(np.argmin(Fpb)); Gb, Fgb = Pb[g].copy(), Fpb[g]
    hist_n, hist_b = [nfev], [Fgb]
    for _ in range(iters):
        r1, r2 = rng.random((swarm, n)), rng.random((swarm, n))
        V = w * V + c1 * r1 * (Pb - X) + c2 * r2 * (Gb - X)     # оновлення швидкості
        X = np.clip(X + V, lo, hi)
        F = np.array([f(x) for x in X]); nfev += swarm
        imp = F < Fpb
        Pb[imp], Fpb[imp] = X[imp], F[imp]
        g = int(np.argmin(Fpb))
        if Fpb[g] < Fgb:
            Gb, Fgb = Pb[g].copy(), Fpb[g]
        hist_n.append(nfev); hist_b.append(Fgb)
    return Gb, Fgb, nfev, np.array(hist_n), np.array(hist_b)

## 5. Головний запуск і графік збіжності

Фіксуємо seed=42 для відтворюваності "головного" запуску кожного алгоритму (вимога завдання — завжди фіксувати початкове значення генератора). Графік збіжності показує найкраще знайдене значення f-f* від кількості обчислень функції (лог. шкала), щоб порівняти швидкість збіжності на кілька порядків величини одночасно.

In [ ]:
SEED_MAIN = 42
rng = np.random.default_rng(SEED_MAIN)
x_sa, f_sa, nfev_sa, hn_sa, hb_sa = simulated_annealing(f, LO, HI, DIM, ALPHA, rng)
rng = np.random.default_rng(SEED_MAIN)
x_ga, f_ga, nfev_ga, hn_ga, hb_ga = genetic(f, LO, HI, DIM, POP, rng)
rng = np.random.default_rng(SEED_MAIN)
x_pso, f_pso, nfev_pso, hn_pso, hb_pso = pso(f, LO, HI, DIM, POP, rng)

print(f"{'Метод':<22}{'f(x*)':>14}{'обчисл. f':>12}")
for name, fv, nf in [("Імітація відпалу", f_sa, nfev_sa),
                      ("Генетичний алгоритм", f_ga, nfev_ga),
                      ("Рій частинок", f_pso, nfev_pso)]:
    print(f"{name:<22}{fv:>14.6e}{nf:>12}")

In [ ]:
EPS_LOG = 1e-12
fig_conv, axcv = plt.subplots(figsize=(8, 5))
for name, hn, hb in [
    ("Імітація відпалу", hn_sa, hb_sa),
    ("Генетичний алгоритм", hn_ga, hb_ga),
    ("Рій частинок", hn_pso, hb_pso),
]:
    shifted = np.maximum(hb - F_STAR, EPS_LOG)
    axcv.semilogy(hn, shifted, label=name, lw=1.6)
axcv.set_xlabel("кількість обчислень f"); axcv.set_ylabel("f(найкращий) - f*  (лог. шкала)")
axcv.set_title(f"Варіант {VARIANT}: збіжність алгоритмів (seed={SEED_MAIN})")
axcv.legend()
fig_conv.tight_layout()
fig_conv.savefig("convergence_base.png", dpi=150)
fig_conv

## 6. Статистика за 20 незалежних запусків

Метаевристики стохастичні — один запуск нічого не доводить. Виконуємо 20 незалежних запусків кожного алгоритму (seed = 0..19) і обчислюємо середнє, стандартне відхилення, найкраще й найгірше значення та частку запусків, що знайшли околицю глобального мінімуму (|f - f*| < 0.5).

In [ ]:
N_RUNS = 20
stats_raw = {"Імітація відпалу": [], "Генетичний алгоритм": [], "Рій частинок": []}
t0 = time.perf_counter()
for seed in range(N_RUNS):
    rng = np.random.default_rng(seed)
    _, fb, *_ = simulated_annealing(f, LO, HI, DIM, ALPHA, rng)
    stats_raw["Імітація відпалу"].append(fb)
    rng = np.random.default_rng(seed)
    _, fb, *_ = genetic(f, LO, HI, DIM, POP, rng)
    stats_raw["Генетичний алгоритм"].append(fb)
    rng = np.random.default_rng(seed)
    _, fb, *_ = pso(f, LO, HI, DIM, POP, rng)
    stats_raw["Рій частинок"].append(fb)
print(f"20x3 незалежних запуски зайняли {time.perf_counter() - t0:.1f} с")

stats_rows = []
for name, vals in stats_raw.items():
    vals = np.array(vals)
    success = int(np.sum(np.abs(vals - F_STAR) < SUCCESS_TOL))
    stats_rows.append(dict(method=name, mean=vals.mean(), std=vals.std(),
                            best=vals.min(), worst=vals.max(),
                            success=success, success_rate=success / N_RUNS))
stats_df = pd.DataFrame(stats_rows)
stats_df

## 7. Дослідження впливу параметрів

Три міні-експерименти (один запуск на значення параметра, seed=42): вплив коефіцієнта охолодження α на імітацію відпалу, вплив розміру популяції на генетичний алгоритм, вплив коефіцієнта інерції w на рій частинок.

In [ ]:
fig_par, axp = plt.subplots(1, 3, figsize=(16, 4.8))

alphas_study = sorted(set([0.85, 0.90, 0.95, 0.99, ALPHA]))
for a in alphas_study:
    rng = np.random.default_rng(SEED_MAIN)
    _, fb, nf, hn, hb = simulated_annealing(f, LO, HI, DIM, a, rng)
    axp[0].semilogy(hn, np.maximum(hb - F_STAR, EPS_LOG), label=f"alpha={a}")
axp[0].set_title("Імітація відпалу: вплив alpha"); axp[0].set_xlabel("обчисл. f"); axp[0].legend(fontsize=8)

pops_study = sorted(set([10, 20, 40, POP]))
for p in pops_study:
    rng = np.random.default_rng(SEED_MAIN)
    _, fb, nf, hn, hb = genetic(f, LO, HI, DIM, p, rng)
    axp[1].semilogy(hn, np.maximum(hb - F_STAR, EPS_LOG), label=f"pop={p}")
axp[1].set_title("ГА: вплив розміру популяції"); axp[1].set_xlabel("обчисл. f"); axp[1].legend(fontsize=8)

for wv in [0.4, 0.72, 0.9]:
    rng = np.random.default_rng(SEED_MAIN)
    _, fb, nf, hn, hb = pso(f, LO, HI, DIM, POP, rng, w=wv)
    axp[2].semilogy(hn, np.maximum(hb - F_STAR, EPS_LOG), label=f"w={wv}")
axp[2].set_title("PSO: вплив коефіцієнта інерції w"); axp[2].set_xlabel("обчисл. f"); axp[2].legend(fontsize=8)

fig_par.suptitle(f"Варіант {VARIANT}: дослідження впливу параметрів (seed={SEED_MAIN})")
fig_par.tight_layout()
fig_par.savefig("param_study_base.png", dpi=150)
fig_par

## 8. Задача комівояжера: жадібний алгоритм проти імітації відпалу з 2-opt

Генеруємо випадкові міста, розв'язуємо жадібним алгоритмом "найближчий сусід" і імітацією відпалу з операцією 2-opt (реверс відрізка маршруту), порівнюємо довжини обох маршрутів.

In [ ]:
N_CITIES = 25
rng_tsp = np.random.default_rng(7)
pts = rng_tsp.uniform(0, 100, (N_CITIES, 2))
Dm = np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=2)
tour_len = lambda t: float(Dm[t, np.roll(t, -1)].sum())


def tsp_greedy_nn(Dm, n):
    tour = [0]
    while len(tour) < n:
        last = tour[-1]
        nxt = min((d, j) for j, d in enumerate(Dm[last]) if j not in tour)[1]
        tour.append(nxt)
    return np.array(tour)


def tsp_sa_2opt(Dm, n, rng, T0=100.0, alpha=0.999, iters=60000):
    t = rng.permutation(n); best, fbest = t.copy(), tour_len(t)
    cur, fcur, T = t.copy(), fbest, T0
    for _ in range(iters):
        i, j = sorted(rng.choice(n, 2, replace=False))
        cand = cur.copy()
        cand[i:j + 1] = cand[i:j + 1][::-1]  # 2-opt: реверс відрізка маршруту
        fc = tour_len(cand)
        if fc < fcur or rng.random() < np.exp(-(fc - fcur) / max(T, 1e-12)):
            cur, fcur = cand, fc
        if fcur < fbest:
            best, fbest = cur.copy(), fcur
        T *= alpha
    return best, fbest


t_greedy = tsp_greedy_nn(Dm, N_CITIES)
f_greedy = tour_len(t_greedy)
rng_tsp2 = np.random.default_rng(7)
t_sa, f_tsp_sa = tsp_sa_2opt(Dm, N_CITIES, rng_tsp2)
tsp_gain = 100 * (f_greedy - f_tsp_sa) / f_greedy
print(f"Жадібний: {f_greedy:.2f}, відпал+2-opt: {f_tsp_sa:.2f}, виграш: {tsp_gain:.1f}%")

fig_tsp, axt = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, tour, name, length in [
    (axt[0], t_greedy, "Жадібний (найближчий сусід)", f_greedy),
    (axt[1], t_sa, "Імітація відпалу + 2-opt", f_tsp_sa),
]:
    order = list(tour) + [tour[0]]
    ax.plot(pts[order, 0], pts[order, 1], "o-", ms=6, lw=1.3, color="#4a7fb5")
    ax.plot(pts[0, 0], pts[0, 1], "rs", ms=9, label="старт")
    ax.set_title(f"{name}\nдовжина = {length:.1f}")
    ax.legend(fontsize=8)
fig_tsp.suptitle(f"ЗКТ, {N_CITIES} міст: жадібний {f_greedy:.1f} vs відпал+2-opt {f_tsp_sa:.1f}")
fig_tsp.tight_layout()
fig_tsp.savefig("tsp_base.png", dpi=150)
fig_tsp

## 9. Оцінка межі практичної неможливості повного перебору

При швидкості 10^9 варіантів за секунду оцінюємо, при якій кількості міст повний перебір (n-1)!/2 маршрутів стає практично неможливим.

In [ ]:
BRUTE_RATE = 1e9
AGE_UNIVERSE_S = 4.35e17  # ~ 13.8 млрд років
rows = []
for n_c in [10, 12, 15, 18, 20, 25, 30]:
    variants_count = math.factorial(n_c - 1) / 2
    seconds = variants_count / BRUTE_RATE
    rows.append(dict(cities=n_c, variants=variants_count, seconds=seconds,
                      fraction_of_universe_age=seconds / AGE_UNIVERSE_S))
pd.DataFrame(rows)